# 🔧 Notebook 2: Integrating External Tools
### Phase 3 – Advanced LangChain Concepts

---

## What are Tools in LangChain?

A **Tool** is any Python function wrapped so that an agent can call it.  
LangChain has two kinds:

1. **Built-in Tools** – Pre-built by LangChain (`DuckDuckGoSearchRun`, `WikipediaQueryRun`, etc.)
2. **Custom Tools** – Any function you write, wrapped with the `Tool` class

```python
from langchain.tools import Tool

my_tool = Tool(
    name='tool_name',        # agent uses this name to call it
    func=my_function,        # the actual Python function
    description='...'        # critical: tells agent WHEN to use it
)
```

## Category 1: Web Search (No API Key Needed)


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

# ── DuckDuckGo Search ────────────────────────────────────────────────────────
# Free, no API key required, great for demos
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import Tool

search = DuckDuckGoSearchRun()
search_tool = Tool(
    name='web_search',
    func=search.run,
    description=(
        'Searches the web using DuckDuckGo. Use for current events, '
        'recent news, or any fact not in training data. '
        'Input: a search query string.'
    )
)

# Test it directly
result = search_tool.run('LangChain latest features 2024')
print(result[:500])

## Category 2: Wikipedia Tool

Wikipedia is ideal for factual, encyclopedic knowledge.  
The `WikipediaAPIWrapper` fetches article summaries.


In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wiki = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=1,           # number of articles to retrieve
        doc_content_chars_max=500  # truncate to this length
    )
)

print(wiki.run('Artificial Intelligence history'))

## Category 3: Custom Python Tools

Any Python function can become a tool. This is the most powerful pattern.


In [ ]:
import math
from datetime import datetime

# ── Calculator Tool ──────────────────────────────────────────────────────────
def calculate(expression: str) -> str:
    """Evaluate a math expression safely."""
    try:
        safe_ns = {'sqrt': math.sqrt, 'pi': math.pi, 'e': math.e}
        return f"Result: {eval(expression, {'__builtins__': {}}, safe_ns)}"
    except Exception as ex:
        return f"Error: {ex}"

calc_tool = Tool(
    name='calculator',
    func=calculate,
    description='Evaluates math expressions. Input: e.g. "sqrt(144) + 2**3"'
)

print(calc_tool.run('sqrt(144) + 2**3'))  # Should print 20.0

# ── Datetime Tool ────────────────────────────────────────────────────────────
def get_datetime(_='') -> str:
    return f"Current date/time: {datetime.now().strftime('%A, %B %d, %Y %I:%M %p')}"

dt_tool = Tool(
    name='current_datetime',
    func=get_datetime,
    description='Returns current date and time. Input: empty string.'
)

print(dt_tool.run(''))

## Category 4: Database Integration (SQLite)

Connect an agent to a SQL database so it can query structured data.


In [ ]:
from sqlalchemy import create_engine, text

# Create an in-memory SQLite database
engine = create_engine('sqlite:///:memory:')

# Seed with sample data
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE products (
            id INTEGER PRIMARY KEY,
            name TEXT,
            price REAL,
            category TEXT
        )
    """))
    conn.execute(text("""
        INSERT INTO products VALUES
        (1, 'Laptop', 999.99, 'Electronics'),
        (2, 'Phone', 699.99, 'Electronics'),
        (3, 'Book', 29.99, 'Education'),
        (4, 'Headphones', 149.99, 'Electronics')
    """))
    conn.commit()

def query_database(sql: str) -> str:
    """Execute a SQL query and return the results."""
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql))
            rows = result.fetchall()
            return str(rows) if rows else 'No results found.'
    except Exception as ex:
        return f'SQL Error: {ex}'

db_tool = Tool(
    name='database_query',
    func=query_database,
    description=(
        'Queries the products SQLite database. '
        'Input: a valid SQL SELECT statement. '
        'Tables: products(id, name, price, category)'
    )
)

# Test: Find all electronics
result = db_tool.run("SELECT name, price FROM products WHERE category = 'Electronics'")
print('Electronics products:', result)

## Category 5: Calling External APIs

Wrap any REST API as a LangChain tool to give agents real-time data access.


In [ ]:
import requests

def get_crypto_price(coin: str) -> str:
    """
    Fetch current cryptocurrency price from CoinGecko (free, no API key).
    Input: coin id, e.g. 'bitcoin', 'ethereum'
    """
    try:
        url = f'https://api.coingecko.com/api/v3/simple/price'
        params = {'ids': coin.lower(), 'vs_currencies': 'usd'}
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        if coin.lower() in data:
            price = data[coin.lower()]['usd']
            return f'Current {coin} price: ${price:,.2f} USD'
        return f'Could not find price for {coin}'
    except Exception as ex:
        return f'API Error: {ex}'

crypto_tool = Tool(
    name='crypto_price',
    func=get_crypto_price,
    description=(
        'Fetches real-time cryptocurrency prices from CoinGecko. '
        'Input: coin name, e.g. bitcoin, ethereum, dogecoin'
    )
)

print(crypto_tool.run('bitcoin'))

## Putting All Tools Together in an Agent


In [ ]:
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain_groq import ChatGroq

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.3, groq_api_key=os.getenv('GROQ_API_KEY'))

# Give agent access to all tools
all_tools = [search_tool, calc_tool, dt_tool, db_tool, crypto_tool]

prompt = hub.pull('hwchase17/react')
agent = create_react_agent(llm=llm, tools=all_tools, prompt=prompt)
executor = AgentExecutor(agent=agent, tools=all_tools, verbose=True,
                         max_iterations=5, handle_parsing_errors=True)

# Test: agent should choose the calculator tool
result = executor.invoke({'input': 'What is the square root of 2025?'})
print('\nAnswer:', result['output'])

## Tools Summary

| Tool | Requires API Key? | Use Case |
|---|---|---|
| DuckDuckGo Search | ❌ No | Web search, current events |
| Wikipedia | ❌ No | Encyclopedic knowledge |
| Custom Python | ❌ No | Calculation, formatting |
| SQLite Database | ❌ No | Structured data queries |
| Google Search API | ✅ Yes | Premium web search |
| OpenAI APIs | ✅ Yes | Embeddings, vision, etc. |
| CoinGecko API | ❌ No | Crypto prices (real-time) |

**👉 Next:** See `03_memory.ipynb` to learn how agents remember context.
